In [1]:
!pip install pyspark

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.sql.functions import col, when, lit
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

make the spark session

In [4]:
#make the spark session
spark = SparkSession.builder.appName("HeartDisease_Pipeline").getOrCreate()

In [5]:
#skema and ingestion
schema = StructType([
    StructField("age", DoubleType(), True),
    StructField("sex", DoubleType(), True),
    StructField("cp", DoubleType(), True),
    StructField("trestbps", DoubleType(), True),
    StructField("chol", DoubleType(), True),
    StructField("fbs", DoubleType(), True),
    StructField("restecg", DoubleType(), True),
    StructField("thalach", DoubleType(), True),
    StructField("exang", DoubleType(), True),
    StructField("oldpeak", DoubleType(), True),
    StructField("slope", DoubleType(), True),
    StructField("ca", StringType(), True),
    StructField("thal", StringType(), True),
    StructField("target", DoubleType(), True)
])

main dataset (heart disease)

In [7]:
df_heart_raw = spark.read.csv("/content/processed.cleveland.data", header=False, schema=schema)
df_heart_raw.show(5)
df_heart_raw.printSchema()

+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
| age|sex| cp|trestbps| chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
|63.0|1.0|1.0|   145.0|233.0|1.0|    2.0|  150.0|  0.0|    2.3|  3.0|0.0| 6.0|   0.0|
|67.0|1.0|4.0|   160.0|286.0|0.0|    2.0|  108.0|  1.0|    1.5|  2.0|3.0| 3.0|   2.0|
|67.0|1.0|4.0|   120.0|229.0|0.0|    2.0|  129.0|  1.0|    2.6|  2.0|2.0| 7.0|   1.0|
|37.0|1.0|3.0|   130.0|250.0|0.0|    0.0|  187.0|  0.0|    3.5|  3.0|0.0| 3.0|   0.0|
|41.0|0.0|2.0|   130.0|204.0|0.0|    2.0|  172.0|  0.0|    1.4|  1.0|0.0| 3.0|   0.0|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
only showing top 5 rows
root
 |-- age: double (nullable = true)
 |-- sex: double (nullable = true)
 |-- cp: double (nullable = true)
 |-- trestbps: double (nullable = true)
 |-- chol: double (nullable = true)
 |-- fbs: double 

reference table (ICD-10)

In [32]:
df_icd = spark.createDataFrame([
    ("I25.1", "Atherosclerotic heart disease"),
    ("Z00.0", "General medical examination"),
    ], ['condition_id', 'icd10_desc'])

data transformation and cleaning

In [33]:
df_heart_clean = df_heart_raw.replace('?', None).na.drop() \
    .withColumn("ca", col("ca").cast(DoubleType())) \
    .withColumn("thal", col("thal").cast(DoubleType()))

df_heart_clean.show(5)

+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
| age|sex| cp|trestbps| chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
|63.0|1.0|1.0|   145.0|233.0|1.0|    2.0|  150.0|  0.0|    2.3|  3.0|0.0| 6.0|   0.0|
|67.0|1.0|4.0|   160.0|286.0|0.0|    2.0|  108.0|  1.0|    1.5|  2.0|3.0| 3.0|   2.0|
|67.0|1.0|4.0|   120.0|229.0|0.0|    2.0|  129.0|  1.0|    2.6|  2.0|2.0| 7.0|   1.0|
|37.0|1.0|3.0|   130.0|250.0|0.0|    0.0|  187.0|  0.0|    3.5|  3.0|0.0| 3.0|   0.0|
|41.0|0.0|2.0|   130.0|204.0|0.0|    2.0|  172.0|  0.0|    1.4|  1.0|0.0| 3.0|   0.0|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
only showing top 5 rows


In [36]:
#binarization target and mapping ICD-0
df_heart_clean = df_heart_clean.withColumn("label", when(col("target") > 0, 1.0). otherwise(0.0))
df_heart_mapped = df_heart_clean.withColumn("condition_id", when(col('label') == 1.0, lit("I25.1")).otherwise(lit("Z00.0")))

In [37]:
#join data
df_final = df_heart_mapped.join(df_icd, on='condition_id', how="left")

In [38]:
print('data pipeline')
df_final.select("age", "sex", "chol", "label", "icd10_desc").show(5)

data pipeline
+----+---+-----+-----+--------------------+
| age|sex| chol|label|          icd10_desc|
+----+---+-----+-----+--------------------+
|63.0|1.0|233.0|  0.0|General medical e...|
|37.0|1.0|250.0|  0.0|General medical e...|
|41.0|0.0|204.0|  0.0|General medical e...|
|56.0|1.0|236.0|  0.0|General medical e...|
|67.0|1.0|286.0|  1.0|Atherosclerotic h...|
+----+---+-----+-----+--------------------+
only showing top 5 rows


In [39]:
#data storage

df_final.write.format('parquet').mode('overwrite').save('heart_disease_lakehouse')

In [40]:
#ai algo random forest
feat_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'thalach', 'oldpeak', "ca", 'thal']
assembler = VectorAssembler(inputCols=feat_cols, outputCol="features")

In [43]:
df_ml = assembler.transform(df_final).select('features', 'label')
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)
# print(train_data.size())

In [44]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, seed=42)
model = rf.fit(train_data)
pred = model.transform(test_data)

In [47]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("Accuracy:", accuracy)

Accuracy: 0.8478260869565217
